# 02 · Etiquetado por reglas: intención, categoría y urgencia (pandas + PyArrow)

**Objetivo:** asignar a cada tweet de **cliente** tres etiquetas (más una bandera de frustración) con diccionarios de palabras, y generar una muestra para etiquetar a mano y medir qué tan buenas son las reglas.

**Etiquetas**
| Columna | Valores |
|---|---|
| `intencion` | `falla`, `solicitud`, `otro` (la falla **no** requiere signo de pregunta ni su ausencia) |
| `categoria` | telefonía/llamadas, cuenta/iCloud, pagos, servicios Apple, conectividad, apps/funciones, iOS/software/rendimiento, hardware, dispositivo_general, otros |
| `urgencia` | `baja`, `media`, `critica` (solo las fallas pueden ser media o crítica) |
| `frustracion` | verdadero/falso (no cambia la urgencia; se guarda para análisis) |

**Archivos que se generan**
| Archivo | Para qué sirve |
|---|---|
| `clientes_etiquetados.parquet` | **Todos** los tweets de clientes con la etiqueta automática. |
| `muestra_para_etiquetar.csv` | Muestra pequeña **sin** etiquetas automáticas, para que tú pongas la etiqueta correcta a mano. |
| `muestra_clave_reglas.csv` | Los mismos tweets **con** la etiqueta automática; se usa solo al final para comparar. |

**Cómo se eligen las categorías:** el código **no las descubre**: salen del catálogo que se define en la sección 2. La sección 6 sirve para comprobar si falta alguna.

## 1. Lectura Parquet y rutas
Los datos de entrada son el archivo Parquet generado por el notebook 01. La lectura y la clasificación se hacen con pandas y PyArrow, sin Spark ni Hadoop.

In [1]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report

IN = Path("../data/processed/apple_support_filtrado.parquet")
OUT = Path("../data/processed/clientes_etiquetados.parquet")
MUESTRA = Path("../data/processed/muestra_para_etiquetar.csv")
CLAVE = Path("../data/processed/muestra_clave_reglas.csv")

## 2. Catálogo de palabras (la parte que tú ajustas)
Todo el conocimiento del clasificador está en estas listas de **expresiones regulares**. Para editarlas basta con agregar o quitar elementos.

**Cómo leerlas**
- `patron()` une una lista en una sola búsqueda que ignora mayúsculas y exige que la palabra **empiece** en un límite de palabra: `crash` detecta `crashes`, pero `bug` no se activa dentro de otra palabra. Para exigir que **termine** también, se agrega `\b` (por ejemplo `apps?\b`, para no confundir "app" con "apple").
- `'?` significa "apóstrofe opcional" (`can'?t` acepta *cant* y *can't*).

**Cómo se decide cada etiqueta**
- **Intención:** si aparece una *falla fuerte* (incluye todo lo crítico) → `falla`. Si no, y hay una solicitud explícita ("how do I…") → `solicitud`. Si no, y hay una *falla débil* ("issue", "restart", "fix") → `falla`. Si no, y hay patrón de pregunta **o** "?" → `solicitud`. Si nada coincide → `otro`. Así "how do I restart my iPhone" es solicitud, pero "my phone won't restart" es falla.
- **Categoría:** gana la categoría con más palabras distintas coincidentes; en empate, la primera del diccionario. Si ninguna coincide pero se nombra un equipo ("iphone", "mac") → `dispositivo_general`; si no → `otros`. Los *síntomas* (glitch, freezing, lag, unresponsive) apuntan a software; los daños físicos (cracked, fallen, battery) a hardware.
- **Urgencia** (solo fallas): **crítica** si hay pérdida de servicio, acceso, datos, dinero, seguridad o función tras daño físico (por ejemplo, no poder hacer llamadas); **media** si hay función degradada; **baja** si es una molestia menor ("weird", "annoying"). Una falla media pasa a **crítica** si hay un *amplificador* de duración, repetición o alcance (*twice*, *constantly*, *since yesterday's update*, *everyone*).

In [2]:
# ---------- 1. INTENCIÓN ----------
# Falla "fuerte": por sí sola indica que algo no funciona
FALLA_FUERTE = [
    r"not working", r"stopped working", r"not able", r"unable", r"won'?t", r"can'?t",
    r"cannot", r"couldn'?t", r"no longer", r"broken", r"crash", r"bug", r"error",
    r"stuck", r"freez", r"frozen", r"glitch", r"lag", r"fail", r"outage", r"down\b",
    r"slow", r"drain", r"dying", r"died", r"dies\b", r"lost", r"missing", r"disappear",
    r"black screen", r"unresponsive", r"soft reset", r"hard reset", r"force restart",
    r"fallen", r"fell\b", r"cracked", r"overheat", r"ruin",
    r"(?:isn|aren|wasn|weren|doesn|don|didn|hasn|haven|hadn)'?t\s+(?:\w+\s+){0,2}?"
    r"(?:allow\w*|let\w*|work\w*|respond\w*|load\w*|show\w*|charg\w*|connect\w*|sync\w*|receiv\w*|send\w*)",
    r"not\s+(?:receiving|sending|syncing|charging|loading|showing|allowing|letting|responding)",
]
# Falla "débil": suele indicar falla, pero puede ser una pregunta ("how do I restart...")
FALLA_DEBIL = [r"issue", r"problem", r"fix", r"restart", r"reboot", r"reset", r"keeps",
               r"annoying", r"weird", r"wrong"]
# Solicitud explícita: gana sobre la falla débil, NO sobre la fuerte
SOLICITUD_EXPLICITA = [r"how (?:do|can|to|does|should)", r"is it possible", r"is there (?:a|any) way"]
# Solicitud general: no requiere signo de pregunta
SOLICITUD = [
    r"where (?:do|can|is|are)", r"what (?:is|does|are|happens)", r"when (?:will|is|does|can)",
    r"is there", r"does (?:anyone|apple|it|the)", r"do (?:you|apple)", r"can (?:i|you|we)",
    r"could (?:you|i)", r"any (?:way|idea|tips?|update|news)", r"please (?:help|advise|explain|confirm)",
    r"how much", r"which", r"wondering", r"want to know", r"looking for", r"need (?:help|info|to know)",
    r"i want to", r"would like to", r"trying to",
]

# ---------- 2. CATEGORÍAS (orden = prioridad en empates) ----------
CATEGORIAS = {
    "telefonia_llamadas": [r"(?:phone |voice )?calls?\b", r"phone calls?", r"voicemail",
                           r"dial", r"ringing", r"no service", r"\bcaller"],
    "cuenta_icloud": [r"icloud", r"apple id", r"password", r"log ?in", r"sign ?in", r"locked",
                      r"verification", r"2fa", r"backup", r"storage", r"account"],
    "pagos_facturacion": [r"charged", r"charges?\b", r"billing", r"\bbills?\b", r"refund",
                          r"payment", r"subscription", r"invoice", r"apple pay",
                          r"credit card", r"purchase"],
    "servicios_apple": [r"apple music", r"imessage", r"facetime", r"siri", r"icloud mail",
                        r"apple tv", r"podcast", r"maps\b", r"itunes", r"airdrop",
                        r"homepod", r"carplay"],
    "conectividad": [r"wi-?fi", r"bluetooth", r"cellular", r"signal", r"network",
                     r"mobile data", r"data\b", r"sim\b", r"carrier", r"roaming",
                     r"hotspot", r"lte\b", r"internet"],
    "apps_funciones": [r"app store", r"apps?\b", r"download", r"testflight", r"safari",
                       r"mail\b", r"photos?\b", r"widget", r"notification", r"feature",
                       r"quick edit", r"edit\b"],
    "ios_software_rendimiento": [r"ios(?:\s?\d+)?\b", r"update", r"upgrade", r"macos", r"beta",
                                 r"install", r"firmware", r"keyboard", r"glitch", r"bug",
                                 r"freez", r"frozen", r"crash", r"lag", r"slow",
                                 r"unresponsive", r"performance"],
    "hardware_dispositivo": [r"battery", r"screens?\b", r"charging", r"charger", r"cracked",
                             r"fallen", r"fell\b", r"dropped", r"water", r"speaker",
                             r"microphone", r"buttons?\b", r"camera", r"overheat",
                             r"turn(?:ing)? on", r"display", r"touch id", r"face id", r"airpods"],
}
# Si hay falla pero ninguna categoría coincide, y solo se nombra el equipo:
DISPOSITIVO = [r"iphone", r"ipad", r"ipod", r"mac(?:book)?\b", r"imac", r"phone",
               r"(?:apple )?watch\b", r"airpods", r"device", r"laptop", r"tablet"]

# ---------- 3. URGENCIA ----------
CRITICA = [
    r"outage", r"down for", r"(?:is|are) down\b", r"services? (?:is |are )?down",
    r"completely (?:dead|broken|unusable|died)", r"unusable", r"bricked",
    r"won'?t (?:turn on|power|boot|start)", r"boot ?loop", r"stuck on (?:the )?(?:apple )?logo",
    r"data loss", r"lost (?:all|everything|my (?:photos|data|contacts|messages))",
    r"(?:photos|data|contacts|messages) (?:gone|deleted|disappeared|lost)",
    r"hacked", r"unauthori[sz]ed", r"stolen", r"fraud", r"locked out",
    r"(?:can'?t|cannot|unable to|not able to)\s+(?:log ?in|sign ?in|access)",
    r"charged (?:twice|\d+)", r"double charged", r"disabled", r"no service", r"911", r"emergency",
    # no puede hacer/recibir llamadas (con hasta 3 palabras entre medio)
    r"(?:not able to|unable to|can'?t|cannot|couldn'?t)\s+(?:\w+\s+){0,3}?(?:make|receive|place|get)\s+(?:\w+\s+){0,4}?calls?",
    # daño físico seguido de pérdida de función
    r"(?:fell|fallen|dropped|water|wet)[^.!?]{0,40}(?:stopped working|dead|not working|won'?t turn on)",
]
# Media = cualquier falla fuerte + estas (menos "annoying/weird/wrong", que quedan en baja)
MEDIA_EXTRA = [r"problem", r"issue", r"fix", r"keeps", r"restart", r"reboot", r"reset"]
AMPLIFICADORES = [
    r"urgent", r"asap", r"immediately", r"still\b", r"for (?:days|hours|weeks|months)",
    r"since (?:yesterday|monday|tuesday|wednesday|thursday|friday|this morning|last|the update)",
    r"(?:past|last) (?:\w+ ){0,2}(?:hours?|days?|weeks?)", r"twice", r"(?:three|four|several|multiple) times",
    r"again", r"constantly", r"every (?:time|day|hour)", r"all the time", r"everyone",
    r"anyone else", r"all (?:users|of us)", r"for work", r"work(?:ing)? from", r"business",
]
# Señal de frustración: NO cambia la urgencia; se guarda aparte para análisis
FRUSTRACION = [r"fuck", r"shit", r"damn", r"crap", r"ridiculous", r"unacceptable", r"terrible",
               r"worst", r"horrible", r"frustrat", r"angry", r"switch(?:ing)? to",
               r"never (?:buy|again)", r"🤬", r"😡", r"😤"]

def patron(lista):
    """Une una lista de expresiones en un solo patrón (sin distinguir mayúsculas)."""
    return r"(?i)\b(?:" + "|".join(lista) + r")"

RX_FALLA_FUERTE = patron(FALLA_FUERTE + CRITICA)   # todo lo crítico también es falla
RX_FALLA_DEBIL = patron(FALLA_DEBIL)
RX_SOL_EXPLICITA = patron(SOLICITUD_EXPLICITA)
RX_SOLICITUD = patron(SOLICITUD)
RX_CRITICA = patron(CRITICA)
RX_MEDIA = patron(FALLA_FUERTE + MEDIA_EXTRA)
RX_AMPL = patron(AMPLIFICADORES)
RX_FRUSTRACION = patron(FRUSTRACION)
RX_DISPOSITIVO = patron(DISPOSITIVO)

## 3. Función de clasificación en pandas
`clasificar()` aplica las mismas reglas a columnas de pandas. Cada patrón aporta como máximo un punto por categoría; en caso de empate, gana la primera categoría del catálogo.

In [3]:
def clasificar(df, columna="texto_limpio"):
    resultado = df.copy()
    t = (resultado[columna].fillna("").astype(str)
         .str.replace(r"[’‘`]", "'", regex=True))

    fuerte = t.str.contains(RX_FALLA_FUERTE, na=False)
    debil = t.str.contains(RX_FALLA_DEBIL, na=False)
    explicita = t.str.contains(RX_SOL_EXPLICITA, na=False)
    solicitud = t.str.contains(RX_SOLICITUD, na=False) | t.str.contains("?", regex=False, na=False)

    resultado["intencion"] = "otro"
    resultado.loc[solicitud, "intencion"] = "solicitud"
    resultado.loc[debil, "intencion"] = "falla"
    resultado.loc[explicita, "intencion"] = "solicitud"
    resultado.loc[fuerte, "intencion"] = "falla"

    puntajes = pd.DataFrame(index=resultado.index)
    for categoria, patrones in CATEGORIAS.items():
        puntajes[categoria] = sum(
            t.str.contains(patron([p]), na=False).astype("int8")
            for p in patrones
        )
    maximo = puntajes.max(axis=1)
    categoria = puntajes.idxmax(axis=1)
    sin_coincidencia = maximo.eq(0)
    dispositivo = t.str.contains(RX_DISPOSITIVO, na=False)
    categoria = categoria.where(
        ~sin_coincidencia,
        dispositivo.map({True: "dispositivo_general", False: "otros"}),
    )
    resultado["categoria"] = categoria

    es_falla = resultado["intencion"].eq("falla")
    media = t.str.contains(RX_MEDIA, na=False)
    critica = t.str.contains(RX_CRITICA, na=False)
    amplificada = t.str.contains(RX_AMPL, na=False)
    resultado["urgencia"] = "baja"
    resultado.loc[es_falla & media & ~amplificada, "urgencia"] = "media"
    resultado.loc[es_falla & (critica | (media & amplificada)), "urgencia"] = "critica"
    resultado["frustracion"] = t.str.contains(RX_FRUSTRACION, na=False)
    return resultado

## 4. Prueba con ejemplos
Antes de correr todo el dataset, se verifica el catálogo con ejemplos reales (los que me compartiste) y casos de contraste. **Agrega aquí tus propios casos difíciles** y vuelve a correr la celda cada vez que cambies el catálogo; es la mejor forma de no romper algo al corregir otra cosa.

In [4]:
ejemplos = [
 "@AppleSupport my phone is not able to receive or make any phone calls.",
 "@AppleSupport iphone glitch",
 "@AppleSupport mac glitch",
 "I've had to soft reset my phone twice in the past two hours because it became unresponsive. @AppleSupport",
 "@AppleSupport please fix the fucking problem with keyboard or I'll switch to @USER",
 "@AppleSupport since yesterdays update, my phone is dying of glitches and constantly freezing...",
 "@AppleSupport extremely frustrating (and coincidental??) that my phone slows way down with the new updates...since the NEW phones are out🤬",
 "@AppleSupport It‘s fallen down and then stopped working",
 "Hey @AppleSupport the iOS update is ruining my Apple iPhone experience pls fix",
 "@AppleSupport No. And it also isn't allowing me to do the new quick edit thing. Hasn't since the first time.",
 # contrastes
 "@AppleSupport How do I restart my iPhone?",
 "@AppleSupport how to change my apple id password",
 "@AppleSupport I was charged twice for my subscription",
 "@AppleSupport thanks for the help!",
]
prueba = pd.DataFrame({"texto_limpio": ejemplos})
pd.set_option("display.max_colwidth", 70)
clasificar(prueba)

,texto_limpio,intencion,categoria,urgencia,frustracion
0,@AppleSupport my phone is not able to receive or make any phone ca...,falla,telefonia_llamadas,critica,False
1,@AppleSupport iphone glitch,falla,ios_software_rendimiento,media,False
2,@AppleSupport mac glitch,falla,ios_software_rendimiento,media,False
3,I've had to soft reset my phone twice in the past two hours becaus...,falla,ios_software_rendimiento,critica,False
4,@AppleSupport please fix the fucking problem with keyboard or I'll...,falla,ios_software_rendimiento,media,True
5,"@AppleSupport since yesterdays update, my phone is dying of glitch...",falla,ios_software_rendimiento,critica,False
6,@AppleSupport extremely frustrating (and coincidental??) that my p...,falla,ios_software_rendimiento,media,True
7,@AppleSupport It‘s fallen down and then stopped working,falla,hardware_dispositivo,critica,False
8,Hey @AppleSupport the iOS update is ruining my Apple iPhone experi...,falla,ios_software_rendimiento,media,False
9,@AppleSupport No. And it also isn't allowing me to do the new quic...,falla,apps_funciones,media,False


## 5. Aplicar las reglas a todos los tweets de clientes
Se leen los tweets de clientes del Parquet, se clasifican y se guarda el resultado completo (también en Parquet).

In [5]:
df = pq.read_table(IN, columns=["tweet_id", "rol", "texto_limpio"]).to_pandas()
cl = df.loc[df["rol"].eq("cliente")].copy()
del df
cl = clasificar(cl)

OUT.parent.mkdir(parents=True, exist_ok=True)
cl.to_parquet(OUT, index=False, engine="pyarrow", compression="snappy")
print("Tweets de clientes etiquetados:", len(cl))

def distribucion(df, col):
    conteos = df[col].value_counts(dropna=False)
    return pd.DataFrame({"count": conteos, "pct": (conteos / len(df)).round(3)})

distribucion(cl, "intencion")

Tweets de clientes etiquetados: 126113


,count,pct
intencion,,
falla,63485,0.503
otro,41764,0.331
solicitud,20864,0.165


## 6. Distribuciones (tablas para Resultados)
La urgencia solo tiene sentido para las **fallas**, por eso se calcula sobre ese subconjunto. La tabla cruzada muestra qué categorías concentran los casos críticos.

In [6]:
fallas = cl.loc[cl["intencion"].eq("falla")].copy()
print("Urgencia (solo fallas):")
display_urg = distribucion(fallas, "urgencia")
print(display_urg, "\n")
print("Categoría (solo fallas):")
print(distribucion(fallas, "categoria"), "\n")
pd.crosstab(fallas["categoria"], fallas["urgencia"])

Urgencia (solo fallas):
          count    pct
urgencia              
media     49969  0.787
critica   11723  0.185
baja       1793  0.028 

Categoría (solo fallas):
                          count    pct
categoria                             
ios_software_rendimiento  26165  0.412
otros                     12702  0.200
dispositivo_general        5747  0.091
apps_funciones             5121  0.081
hardware_dispositivo       3693  0.058
cuenta_icloud              2534  0.040
servicios_apple            2356  0.037
conectividad               2201  0.035
telefonia_llamadas         1517  0.024
pagos_facturacion          1449  0.023 



urgencia,baja,critica,media
categoria,,,
apps_funciones,128,1014,3979
conectividad,53,440,1708
cuenta_icloud,63,822,1649
dispositivo_general,292,1035,4420
hardware_dispositivo,88,702,2903
ios_software_rendimiento,381,4901,20883
otros,682,1532,10488
pagos_facturacion,25,314,1110
servicios_apple,63,423,1870


## 7. ¿Las categorías están completas? Cómo comprobarlo
Las categorías son una hipótesis tuya y los datos pueden contradecirla. Dos revisiones:

1. **Cobertura:** qué porcentaje de las fallas quedó en `otros` o `dispositivo_general` (es decir, sin tema identificado). Como guía práctica, conviene que sea bajo (por ejemplo < 15-20 %).
2. **Palabras frecuentes en esas fallas sin tema:** si aparece una palabra repetida (por ejemplo `homepod`, `carplay`), probablemente merece su categoría o entrar a una existente.

In [7]:
sin_tema = fallas.loc[fallas["categoria"].isin(["otros", "dispositivo_general"])]
print(f"Fallas sin tema identificado: {len(sin_tema) / len(fallas):.1%}")

vec = CountVectorizer(stop_words="english", token_pattern=r"(?u)\b[a-zA-Z']{3,}\b")
matriz = vec.fit_transform(sin_tema["texto_limpio"].fillna("").str.lower())
conteos = matriz.sum(axis=0).A1
palabras = (pd.DataFrame({"palabra": vec.get_feature_names_out(), "count": conteos})
            .sort_values("count", ascending=False).head(40))
palabras

Fallas sin tema identificado: 29.1%


,palabra,count
479,applesupport,12347
8878,user,10791
3074,fix,6483
6063,phone,2742
8863,url,2644
4309,iphone,2203
6374,problem,2093
4378,issue,2000
1230,can't,1900
4386,it's,1697


## 8. Ver ejemplos de cada etiqueta
Antes de etiquetar a mano, lee ejemplos por urgencia para decidir si las definiciones tienen sentido. Cambia `nivel` para ver los otros. También puedes cambiar el filtro para revisar una categoría concreta.

In [8]:
nivel = "critica"   # prueba: "media", "baja"
pd.set_option("display.max_colwidth", 200)
(cl.loc[cl["urgencia"].eq(nivel), ["texto_limpio", "categoria", "frustracion"]]
   .sample(n=min(15, cl["urgencia"].eq(nivel).sum()), random_state=3))

,texto_limpio,categoria,frustracion
217944,"@AppleSupport @USER Me think it's a worldwide problem here, @AppleSupport still wondering where the fix is. (Me has to speak like this so me don't get A ?)",otros,False
176834,@AppleSupport Since this morning. I have an Apple leather case. No drops or breaks. I tried restarting the phone and cleaning the lens with a Qtip. I don't see any debris.,dispositivo_general,False
93719,@USER @AppleSupport enough of this stupid I️ symbol. What is wrong with you? Everyone is complaining. Fix the glitch already!,ios_software_rendimiento,False
145118,Panchodo kutteyo kusreyo bastaaro my battery dies so quickly since the update @USER,ios_software_rendimiento,False
227641,"@AppleSupport When my iPhone is connected to my car via bluetooth, I can't use Siri (the mic will not pick me up), so I have to do that little dance every time and tell Siri to use the phone mic. ...",servicios_apple,False
173490,@AppleSupport hey apple i have been having this problem since yesterday please help <URL>,otros,False
10912,@AppleSupport @applesupport - READ THE NEWS -- everyone is complaining about battery life. This isn't an issue that I am having. It's an issue period.,hardware_dispositivo,False
164478,@AppleSupport can't call anyone or receive calls 🙄,telefonia_llamadas,False
229523,"@AppleSupport howdy. My Apple Music hasn't been working the last few days, just keeps saying loading when I try to search for music or go into the browse section. Any idea what it could be? <URL>",servicios_apple,False
162775,Trying to update some #apps on my @USER and im told my apple ID has been disabled....suggestions? @AppleSupport,cuenta_icloud,False


## 9. Muestra para etiquetado manual (a ciegas)
Se toma una muestra **estratificada**: hasta 40 tweets por combinación intención × urgencia, con selección aleatoria reproducible.

Se generan **dos** archivos:
- `muestra_para_etiquetar.csv`: solo `tweet_id`, texto y columnas vacías `*_manual`. **No incluye la etiqueta de las reglas**, para que no te influya.
- `muestra_clave_reglas.csv`: los mismos tweets con la etiqueta automática; se usa en el paso 10.

En `comentario_error` anota por qué un tweet es difícil o ambiguo; esos comentarios alimentan tu subsección de errores y aciertos.

**Valores para etiquetar:** intención `falla`/`solicitud`/`otro`; urgencia `baja`/`media`/`critica`; categoría, los nombres del catálogo, `dispositivo_general` u `otros`.

In [9]:
muestra = (cl.sample(frac=1, random_state=42)
             .groupby(["intencion", "urgencia"], group_keys=False)
             .head(40).copy())

para_etiquetar = muestra[["tweet_id", "texto_limpio"]].copy()
for c in ["intencion_manual", "categoria_manual", "urgencia_manual", "comentario_error"]:
    para_etiquetar[c] = ""

MUESTRA.parent.mkdir(parents=True, exist_ok=True)
para_etiquetar.to_csv(MUESTRA, index=False, encoding="utf-8-sig")
muestra[["tweet_id", "intencion", "categoria", "urgencia"]].to_csv(CLAVE, index=False)
print(len(muestra), "tweets en la muestra")

200 tweets en la muestra


## 10. Evaluación contra tu etiquetado manual (hacer después de etiquetar)
Cuando termines de llenar `muestra_para_etiquetar.csv` (guárdalo como CSV), esta celda compara tus etiquetas con las de las reglas y calcula precisión, sensibilidad y **F1** por clase: la métrica de tu hipótesis (≥ 80 %).

In [10]:
from sklearn.metrics import classification_report

manual = pd.read_csv(MUESTRA, dtype={"tweet_id": str})
clave = pd.read_csv(CLAVE, dtype={"tweet_id": str})
m = manual.merge(clave, on="tweet_id")
m = m[m["intencion_manual"].notna() & (m["intencion_manual"] != "")]

if m.empty:
    print("Aún no hay etiquetas manuales.")
else:
    for dim in ["intencion", "categoria", "urgencia"]:
        print(f"\n=== {dim} ===")
        print(classification_report(m[f"{dim}_manual"], m[dim], zero_division=0))

Aún no hay etiquetas manuales.
